# ☢️ SPECT/CT 기반 3D 병변 자동 분할 대시보드
### [Cell 1] 임상 검증 및 타겟 볼륨(ROI) 확정
임상 의료진은 아래 셀을 실행(`Shift + Enter`)하여 대시보드를 구동하고, 3D 단면 시각화를 통해 병변의 타겟 체적(ml)과 Threshold를 확정해 주십시오.

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.transform import iradon, radon
from scipy.ndimage import gaussian_filter, shift
import os, warnings

warnings.filterwarnings("ignore")
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'text.color': '#333333', 'axes.labelcolor': '#333333',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.edgecolor': '#AAAAAA'
})

print(">>> [Step 1] Initializing Clinical SPECT/CT Dashboard...")

CFG = {
    'SPECT_PATH': './data/SPECT/SPECT-CT_SC001_DS.dcm', 
    'EXPORT_DIR': './MC_Physics_Input',
    'ANGLES': 60, 'ARC': 360.0,
    'FBP_FILTER': 'shepp-logan',
    'OSEM_ITER': 4,
    'MAP_ITER': 15, 'MAP_BETA': 0.02,
    'FLIP_Z': True, 'FLIP_Y': True,
    'COR_OFFSET': 0.0,
    'SCAN_DIR': 'CCW'
}

os.makedirs(CFG['EXPORT_DIR'], exist_ok=True)
ALGO_NAMES = ['FBP', 'OSEM', 'MAP']
CMAPS = ['Greys', 'Blues', 'Reds']

class ReconStudio:
    def __init__(self):
        self.vols = {}; self.meta = {}; self.center = (0,0,0)
        self._run_engine()
        
    def _run_engine(self):
        if not os.path.exists(CFG['SPECT_PATH']):
            print(f"🚨 Error: DICOM 파일을 찾을 수 없습니다: {CFG['SPECT_PATH']}")
            return

        img = sitk.ReadImage(CFG['SPECT_PATH'])
        raw = sitk.GetArrayFromImage(img).astype(np.float32)
        if raw.shape[0] == CFG['ANGLES']: self.sinograms = np.transpose(raw, (1, 0, 2))
        else: self.sinograms = raw
        
        sp = img.GetSpacing()
        vox_ml = (sp[0] * sp[0] * sp[1]) / 1000.0 
        self.meta = {'spacing': (sp[0], sp[0], sp[1]), 'shape': (self.sinograms.shape[0], self.sinograms.shape[2], self.sinograms.shape[2]), 'vox_ml': vox_ml}
        
        d, angles, w = self.sinograms.shape
        theta = np.linspace(0., CFG['ARC'], angles, endpoint=False)[::-1] if CFG['SCAN_DIR'] == 'CW' else np.linspace(0., CFG['ARC'], angles, endpoint=False)
            
        v_fbp, v_osem, v_map = [np.zeros((d, w, w), dtype=np.float32) for _ in range(3)]
        
        print(">>> Reconstructing 3D Volumes (FBP, OSEM, MAP)... Please wait.")
        for z in range(d):
            sino = self.sinograms[z].T 
            if np.max(sino) < 1.0: continue
            if CFG['COR_OFFSET'] != 0.0: sino = shift(sino, [CFG['COR_OFFSET'], 0], mode='nearest')
                
            v_fbp[z] = np.maximum(iradon(sino, theta=theta, filter_name=CFG['FBP_FILTER'], circle=True), 0)
            est = np.ones((w, w), dtype=np.float32)
            sens = iradon(np.ones_like(sino), theta=theta, filter_name=None, circle=True) + 1e-9
            for _ in range(CFG['OSEM_ITER']): est *= (iradon(sino/(radon(est, theta)+1e-9), theta, filter_name=None) / sens)
            v_osem[z] = est
            est_m = est.copy()
            for _ in range(CFG['MAP_ITER']):
                upd = iradon(sino/(radon(est_m, theta)+1e-9), theta, filter_name=None, circle=True)
                pen = CFG['MAP_BETA'] * (est_m - gaussian_filter(est_m, sigma=0.8))
                est_m *= (upd / (sens + pen + 1e-9))
            v_map[z] = est_m

        if CFG['FLIP_Z']: v_fbp, v_osem, v_map = [np.flip(v, axis=0) for v in [v_fbp, v_osem, v_map]]
        if CFG['FLIP_Y']: v_fbp, v_osem, v_map = [np.flip(v, axis=1) for v in [v_fbp, v_osem, v_map]]
        self.vols = {'FBP': v_fbp, 'OSEM': v_osem, 'MAP': v_map}
        self.center = np.unravel_index(np.argmax(v_map), (d, w, w))
        print("✅ Reconstruction Complete.")

    def get_metrics(self, th):
        stats = {}
        for algo in ALGO_NAMES:
            vol = self.vols[algo]
            v_max = np.max(vol)
            mask = (vol / (v_max + 1e-9) * 100.0 > th).astype(np.uint8)
            stats[algo] = {'vol_ml': np.sum(mask) * self.meta['vox_ml'], 'mask': mask, 'max': v_max, 'raw_vol': vol}
        return stats

if 'eng' not in globals(): eng = ReconStudio()
sz, sy, sx = eng.meta['shape']
hz, hy, hx = eng.center
asp_z = eng.meta['spacing'][2] / eng.meta['spacing'][0]

def draw_final_dashboard(z, y, x, th):
    data = eng.get_metrics(th)
    fig = plt.figure(figsize=(22, 11))
    gs = fig.add_gridspec(2, 4, width_ratios=[0.6, 1, 1, 1])
    plt.subplots_adjust(wspace=0.15, hspace=0.25, top=0.88, bottom=0.05, right=0.95)
    v_unit = eng.meta['vox_ml'] * 1000
    fig.suptitle(f"Thyroid Recon Analysis | Z:{z} Y:{y} | {v_unit:.2f} $mm^3$ | Th: {th}%", color='#003366', fontsize=24, fontweight='heavy')

    ax_sag = fig.add_subplot(gs[0, 0])
    ax_sag.imshow(np.flipud(np.max(eng.vols['MAP'], axis=2)), cmap='gray_r', aspect=asp_z)
    ax_sag.axhline(sz - z - 1, c='#00CC00', lw=2); ax_sag.axvline(y, c='#0066CC', lw=2, ls='--')
    ax_sag.set_title("① Sagittal Scout", color='#444444', fontsize=14); ax_sag.axis('off')
    ax_cor = fig.add_subplot(gs[1, 0])
    ax_cor.imshow(np.flipud(np.max(eng.vols['MAP'], axis=1)), cmap='gray_r', aspect=asp_z)
    ax_cor.axhline(sz - z - 1, c='#00CC00', lw=2); ax_cor.axvline(x, c='#FF3300', lw=2, ls='--')
    ax_cor.set_title("② Coronal Scout", color='#444444', fontsize=14); ax_cor.axis('off')

    for i, algo in enumerate(ALGO_NAMES):
        col = i + 1
        vol = eng.vols[algo]
        mask = data[algo]['mask']
        vmax = data[algo]['max']
        
        ax_ax = fig.add_subplot(gs[0, col])
        im = ax_ax.imshow(vol[z], cmap=CMAPS[i], vmin=0, vmax=vmax)
        if np.sum(mask[z]) > 0: ax_ax.contour(mask[z], colors='#FFD700', linewidths=1.5)
        ax_ax.set_title(f"{algo}\n({data[algo]['vol_ml']:.2f} ml)", color='#222222', fontsize=16)
        ax_ax.axvline(x, c='#999999', ls=':', alpha=0.8); ax_ax.axhline(y, c='#999999', ls=':', alpha=0.8)
        if col == 1: ax_ax.set_ylabel("AXIAL (Z)", color='#333333', fontsize=14)
        ax_ax.set_xticks([]); ax_ax.set_yticks([])
        cb = plt.colorbar(im, ax=ax_ax, fraction=0.046, pad=0.04)
        cb.ax.tick_params(labelsize=8, color='#333333', labelcolor='#333333')

        ax_co = fig.add_subplot(gs[1, col])
        ax_co.imshow(np.flipud(vol[:, y, :]), cmap=CMAPS[i], aspect=asp_z, vmin=0, vmax=vmax)
        if np.sum(np.flipud(mask[:, y, :])) > 0: ax_co.contour(np.flipud(mask[:, y, :]), colors='#FFD700', linewidths=1.5)
        ax_co.set_title(f"{algo} (Coronal)", color='#666666', fontsize=12)
        ax_co.axhline(sz - z - 1, c='#999999', ls=':', alpha=0.8); ax_co.axvline(x, c='#999999', ls=':', alpha=0.8)
        if col == 1: ax_co.set_ylabel("CORONAL (Y)", color='#333333', fontsize=14)
        ax_co.set_xticks([]); ax_co.set_yticks([])
    plt.show()

st = {'description_width': 'initial'}
sl_z = widgets.IntSlider(value=hz, min=0, max=sz-1, description='Axial Z', style=st)
sl_y = widgets.IntSlider(value=hy, min=0, max=sy-1, description='Coronal Y', style=st)
sl_x = widgets.IntSlider(value=hx, min=0, max=sx-1, description='Sagittal X', style=st)
sl_th = widgets.FloatSlider(value=35, min=5, max=90, description='Threshold %', style=st)
out = widgets.Output()

def update_ui(z, y, x, th):
    with out:
        clear_output(wait=True)
        draw_final_dashboard(z, y, x, th)

widgets.interactive_output(update_ui, {'z':sl_z, 'y':sl_y, 'x':sl_x, 'th':sl_th})
display(widgets.VBox([widgets.HBox([sl_z, sl_y, sl_x]), sl_th]), out)


### [Cell 2] Monte Carlo Physics Engine Bridge (OpenGATE / PHITS / Pure Geant4)
**[물리학자용]** 임상의가 확정한 Target 체적을 바탕으로 다양한 몬테카를로 엔진(OpenGATE, PHITS, 순수 Geant4 C++)의 입력 파일 및 이진(Binary) 데이터를 자동 생성합니다. 필요하신 엔진의 주석을 해제하여 사용하십시오.

In [ ]:
"""
# ==============================================================================
# [1] OpenGATE Bridge (.mhd)
# ==============================================================================
def export_to_opengate(th_value):
    final_metrics = eng.get_metrics(th_value)
    export_vol = (final_metrics['MAP']['mask'] * 255).astype(np.uint8)
    sitk_img = sitk.GetImageFromArray(export_vol)
    sitk_img.SetSpacing(eng.meta['spacing'])
    out_path = f"{CFG['EXPORT_DIR']}/Target_MAP_GATE.mhd"
    sitk.WriteImage(sitk_img, out_path)
    print(f"✅ OpenGATE Voxelized Source 추출 완료: {out_path}")
# export_to_opengate(sl_th.value)
"""

"""
# ==============================================================================
# [2] PHITS Bridge (.inp / .txt)
# ==============================================================================
def export_to_phits(th_value):
    export_dir = CFG['EXPORT_DIR']
    final_metrics = eng.get_metrics(th_value)
    src_arr = final_metrics['MAP']['raw_vol'] * final_metrics['MAP']['mask'] # 활성도 분포
    
    z_dim, y_dim, x_dim = src_arr.shape
    spacing_mm = eng.meta['spacing']
    dx, dy, dz = spacing_mm[0]/10.0, spacing_mm[1]/10.0, spacing_mm[2]/10.0
    xmin, xmax = -x_dim*dx/2, x_dim*dx/2
    ymin, ymax = -y_dim*dy/2, y_dim*dy/2
    zmin, zmax = -z_dim*dz/2, z_dim*dz/2

    mat_arr = np.ones_like(src_arr, dtype=int) * 2
    np.savetxt(os.path.join(export_dir, 'voxel_mat.txt'), mat_arr.flatten(), fmt='%d')
    
    with open(os.path.join(export_dir, 'voxel_geom.inp'), 'w') as f:
        f.write(f"[ V o x e l ]\n  x-type = 2\n  nx = {x_dim}\n  xmin = {xmin:.4f}\n  xmax = {xmax:.4f}\n")
        f.write(f"  y-type = 2\n  ny = {y_dim}\n  ymin = {ymin:.4f}\n  ymax = {ymax:.4f}\n")
        f.write(f"  z-type = 2\n  nz = {z_dim}\n  zmin = {zmin:.4f}\n  zmax = {zmax:.4f}\n  file = voxel_mat.txt\n")

    z_idx, y_idx, x_idx = np.nonzero(src_arr)
    weights = src_arr[z_idx, y_idx, x_idx]
    tot_weight = np.sum(weights)
    
    with open(os.path.join(export_dir, 'voxel_source.inp'), 'w') as f:
        f.write("[ S o u r c e ]\n  totfact = 1.0\n  proj = isotope\n  nuclide = 131I\n  dir = all\n\n")
        f.write(f"[ M u l t i  S o u r c e ]\n  pegs = {len(weights)}\n")
        for i in range(len(weights)): f.write(f"  wgt({i+1}) = {weights[i]/tot_weight:.6e}\n")
        f.write("\n")
        for i in range(len(weights)):
            x0, x1 = xmin + x_idx[i]*dx, xmin + x_idx[i]*dx + dx
            y0, y1 = ymin + y_idx[i]*dy, ymin + y_idx[i]*dy + dy
            z0, z1 = zmin + z_idx[i]*dz, zmin + z_idx[i]*dz + dz
            f.write(f"  <source> = {i+1}\n    s-type = 2\n")
            f.write(f"    x0 = {x0:.4f}\n    x1 = {x1:.4f}\n    y0 = {y0:.4f}\n    y1 = {y1:.4f}\n    z0 = {z0:.4f}\n    z1 = {z1:.4f}\n\n")
    print(f"✅ PHITS 입력 파일 자동 생성 완료: {export_dir}")
# export_to_phits(sl_th.value)
"""

"""
# ==============================================================================
# [3] Pure Geant4 (C++) Bridge (.bin Binary Dump)
# ==============================================================================
def export_to_geant4(th_value):
    export_dir = CFG['EXPORT_DIR']
    final_metrics = eng.get_metrics(th_value)
    
    # 1. 기하학적 매질 밀도(Density) 구성: 연조직(1.04 g/cm3)으로 가정
    # 실제 환자의 CT Array가 있다면 HU 값을 변환하여 적용합니다.
    density_arr = np.ones_like(final_metrics['MAP']['raw_vol'], dtype=np.float32) * 1.04 
    density_arr[final_metrics['MAP']['mask'] == 0] = 0.001 # Air
    
    # 2. 방사선원 활성도 맵 구성
    source_arr = (final_metrics['MAP']['raw_vol'] * final_metrics['MAP']['mask']).astype(np.float32)
    
    # C++에서 빠르게 파싱할 수 있도록 연속된 이진(Binary) 포맷으로 덤프
    density_arr.tofile(os.path.join(export_dir, 'voxel_density.bin'))
    source_arr.tofile(os.path.join(export_dir, 'voxel_source.bin'))
    print(f"✅ Geant4 C++ 파싱용 이진 파일(Density/Source) 덤프 완료: {export_dir}")

# export_to_geant4(sl_th.value)

# ------------------------------------------------------------------------------
# [Geant4 C++ 핵심 구현 코드 가이드 (물리학자 참조용)]
# ------------------------------------------------------------------------------
# 아래는 생성된 .bin 파일을 읽어 Geant4 엔진에 적재하는 C++ 구조입니다.

/* 
1. [DetectorConstruction.cc]
   G4PhantomParameterisation을 사용하여 Voxel Geometry를 구성합니다.
   
   #include "G4PhantomParameterisation.hh"
   // ...
   G4PhantomParameterisation* param = new G4PhantomParameterisation();
   param->SetVoxelDimensions(dx/2, dy/2, dz/2);
   param->SetNoVoxel(nx, ny, nz);
   
   // std::ifstream으로 "voxel_density.bin"을 읽어 Material 할당
   std::vector<G4Material*> materialsArray;
   // ... 밀도에 따라 G4NistManager에서 매질(G4_TISSUE, G4_AIR)을 불러와 할당 ...
   param->SetMaterials(materialsArray);

2. [PrimaryGeneratorAction.cc]
   수백만 개의 복셀 선원을 G4GeneralParticleSource(GPS)의 매크로로 제어하는 것은 
   연산 효율이 떨어지므로, 누적분포함수(CDF)를 직접 구성하여 샘플링합니다.
   
   // 1) 초기화 시 "voxel_source.bin"을 읽어 1D 벡터에 저장 후 CDF 배열 생성
   std::vector<G4double> cdf_array;
   // ... 누적 합 계산 및 정규화 ...
   
   // 2) 매 이벤트마다 랜덤 복셀 추출 (Binary Search 방식 O(log N))
   G4double rand_val = G4UniformRand();
   auto it = std::lower_bound(cdf_array.begin(), cdf_array.end(), rand_val);
   G4int voxel_id = std::distance(cdf_array.begin(), it);
   
   // 3) 추출된 Voxel의 3D 좌표를 역산(x, y, z)하고 해당 공간 내에 Random 오프셋 추가
   G4ThreeVector position(x_pos, y_pos, z_pos);
   particleGun->SetParticlePosition(position);
   
   // 4) I-131 이온 방출 (G4RadioactiveDecayPhysics 필수 활성화)
   particleGun->SetParticleDefinition(G4IonTable::GetIonTable()->GetIon(53, 131, 0.0));
   particleGun->GeneratePrimaryVertex(anEvent);

3. [PhysicsList.cc & Scoring]
   - 물리 모델: G4EmStandardPhysics_option3 (Medical Physics 표준) 및 G4RadioactiveDecayPhysics 등록
   - 선량 스코어링: G4PSDoseDeposit을 사용하여 Voxel별 흡수 선량(Gy)을 기록하여 최종 3D 분포도 확보
*/
"""
